# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YoussefZaky208/Flyrank-ML-Track-Assignemnt/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring, built on top of a classification sub-model.**

The final deliverable for Lane 2 is a ranked review queue — an ordered list of content items a
reviewer works down from the top, not a single yes/no answer. That makes the *outer* task a
**ranking/scoring** problem (matches "which ones first?" in the framing-ml-problems mapping).

Underneath the ranking, though, I need a per-page number to sort by, and the starter pipeline
builds that number from a **classification** sub-model: predict the probability that a page is
"declining" (or, in a stronger future version, "will decline"), then rank pages by that
probability (optionally blended with a transparent baseline score, as the starter pipeline does:
`final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`).

So: classification produces the *score*; ranking is what the *reviewer actually uses*. I'm
naming both because grading the wrong one would be a mistake — a classifier can have a mediocre
overall accuracy and still produce an excellent top-of-queue ranking, which is exactly what
matters for a capacity-limited reviewer (see Section 3).

In [ ]:
# Section 1 is a qualitative task-type call - no supporting code required here.
# See Section 4 below for the dataframe and Section 5 for evidence the pattern is genuinely tangled.


## 2. Target or proxy

**For now: a proxy label**, inherited from the starter pipeline —

```text
is_declining_label = (trend_direction == "down")
```

This is a **proxy, not an observed future outcome**. `trend_direction` is itself derived from
`trend_pct`, which is computed from the *same* trailing 90-day window as the features. So this
label answers "is this page currently classified as declining by a rule?", not "will this page's
traffic actually be lower a month from now?" That's a meaningfully weaker claim, and I'm keeping
it visible in the code below rather than hiding it.

**The stronger version I want to move to** (once I have the warehouse's daily fact table
available, per the lane guide and the `flyrank-data` skill): a genuinely future-looking label —

```text
features from a prior 90-day window -> decline (or recovery) over the NEXT 30 days
```

built with a strict prior-window/target-window split and a leakage audit before I trust it. I'm
naming this now so Section 5 (leakage) of later notebooks isn't a surprise — it's the single
biggest risk in this lane, and the guide calls it out by name (`docs/ml-intern-dataset-and-lane-guide.md`, §5, §12).

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same eligibility filter as the starter pipeline
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

# The proxy label, computed exactly as the starter pipeline defines it
eligible = eligible.assign(is_declining_label=(eligible["trend_direction"] == "down"))

print(f"Eligible rows: {len(eligible):,}")
print("Proxy label value counts:")
print(eligible["is_declining_label"].value_counts())
print(f"Proxy label base rate (declining=True): {eligible['is_declining_label'].mean():.3f}")
print()
print("Reminder: trend_direction (and therefore this label) is derived from trend_pct,")
print("which is computed from the SAME 90-day window as the features -- a proxy, not a")
print("forward-looking outcome. See the write-up above.")


Eligible rows: 30,000
Proxy label value counts:
is_declining_label
True     16262
False    13738
Name: count, dtype: int64
Proxy label base rate (declining=True): 0.542

Reminder: trend_direction (and therefore this label) is derived from trend_pct,
which is computed from the SAME 90-day window as the features -- a proxy, not a
forward-looking outcome. See the write-up above.


## 3. Success metric

**Primary metric: Precision@50** (matching a reviewer who can realistically open ~50 pages a
week — see Section 2 of `w01_research_question.ipynb` for the capacity argument).

Why this metric and not plain accuracy:
- The queue is only useful at the *top*. A reviewer never scrolls to rank 10,000 — they work
  down from rank 1. A model can be mediocre in the tail and still be excellent where it matters.
- Accuracy is misleading here anyway: the base rate for the proxy label is already 0.542 (see
  code below), so a model could hit ~54% accuracy by guessing the majority class and tell a
  reviewer nothing useful.
- Precision@K also matches how the guide frames threshold-picking (`docs/ml-intern-dataset-and-lane-guide.md`,
  §11): pick the K that matches real review capacity, then judge the method by how clean the
  top of that list actually is.

**Secondary metrics I'll track alongside it:** ROC-AUC and average precision (to see overall
separability, not just the top-K slice) and recall at that same threshold (because a false
negative — a real decliner that never surfaces — compounds silently over time, per the cost
argument in `w01_research_question.ipynb`).

**A number I already have to defend this metric choice:** the starter pipeline's own numbers
(pulled again below) show precision@50 moving from 0.240 (baseline rule) to 0.740 (random
forest) on this same proxy label — a large, measurable gap that a plain accuracy score would
have hidden.

In [ ]:
with open("outputs/model_report.md") as f:
    report = f.read()

in_table = False
print("Starter pipeline's own model comparison (outputs/model_report.md):")
for line in report.splitlines():
    if line.startswith("| Model"):
        in_table = True
    if in_table:
        print(" ", line.strip())
        if in_table and line.strip() == "":
            break

print()
print("Precision@50: baseline_rules=0.240 -> random_forest=0.740")
print("That gap is the number I'm defending as my success metric target for this lane.")


Starter pipeline's own model comparison (outputs/model_report.md):
  | Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
  |---|---:|---:|---:|---:|---:|
  | decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
  | logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
  | random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
  | baseline_rules | 0.627 | 0.468 | 0.240 | - | - |
  

Precision@50: baseline_rules=0.240 -> random_forest=0.740
That gap is the number I'm defending as my success metric target for this lane.


## 4. The unit of analysis, as a real dataframe

**One row = one content item** (`content_id`), using its trailing 90-day metrics, filtered to
the same eligibility rule the starter pipeline uses (`impressions_90d > 0` and
`content_age_days >= 90`, then deduplicated by `content_id` — see
`docs/ml-intern-dataset-and-lane-guide.md`, §5).

In [ ]:
cols = [
    "content_id", "client_id", "impressions_90d", "sessions_90d", "trend_direction",
    "is_declining_label", "days_since_last_update", "avg_position", "ctr",
    "engagement_rate", "content_age_days", "word_count",
]
print("Unit of analysis: one row = one content item (a single pseudonymized page).")
print(f"Shape: {eligible[cols].shape}")
eligible[cols].head(8)


Unit of analysis: one row = one content item (a single pseudonymized page).
Shape: (30000, 12)


,content_id,client_id,impressions_90d,sessions_90d,trend_direction,is_declining_label,days_since_last_update,avg_position,ctr,engagement_rate,content_age_days,word_count
0,content_304f48230142,client_f369cb89fc,3803,17,down,True,20,10.6,0.76,5.88,187,3221.0
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down,True,25,20.3,0.05,0.00,445,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down,True,20,36.5,0.09,0.00,141,3515.0
3,content_331d6c4de07b,client_19581e27de,11751,78,stable,False,22,6.2,0.49,1.28,463,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down,True,14,44.0,0.13,0.00,263,2803.0
5,content_d4084a4bc775,client_f369cb89fc,3970,5,down,True,20,8.5,0.03,0.00,147,3080.0
6,content_9a34b442b552,client_8722616204,20,1,down,True,20,7.0,0.00,0.00,90,3059.0
7,content_a63219c6e95a,client_19581e27de,1724,28,stable,False,22,21.2,0.06,3.57,445,NaN


## 5. Why ML beats a fixed rule here

The reason codes in this lane are not mutually exclusive — the same page routinely trips
several rules at once (see the overlap count below), and a person can't easily hand-weight six
interacting, overlapping conditions into one clean priority order. A model can learn *how much*
each signal matters and how they trade off against each other; a single if-statement can only
say yes/no per rule and can't rank *within* the "yes" group.

Concretely:
- A stale page only matters if it still has demand — staleness and impressions interact.
- A "declining" page with tiny impressions is much less urgent than a "declining" page with
  massive impressions — the rule treats both the same; a model can weigh them differently.
- Multiple rules can fire on the same row (see the overlap table below) — a plain rule set can
  flag a page as a "match" but has no principled way to say row A is 20% more urgent than row B
  when both match 3 out of 6 conditions.

This is exactly the situation the `framing-ml-problems` skill describes: "ML earns its place
only when the pattern is real but too messy to write by hand — many signals, tangled, shifting
over time." The evidence that it *does* pay off here isn't hypothetical — it's the 0.240 -> 0.740
precision@50 jump from the starter pipeline, reprinted below.

In [ ]:
# Evidence the pattern is genuinely tangled: how often do multiple reason-code
# rules fire on the SAME row? (rules as defined in docs/ml-intern-dataset-and-lane-guide.md, section 5)

stale_visible = (eligible["days_since_last_update"] >= 180) & (eligible["impressions_90d"] >= 500)
declining_with_demand = (eligible["trend_direction"] == "down") & (eligible["impressions_90d"] >= 100)
thin_visible = (eligible["word_count"] > 0) & (eligible["word_count"] < 1200) & (eligible["impressions_90d"] >= 250)
page_one_decay_risk = (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 10) & (eligible["content_age_days"] >= 180)
low_ctr_visible = (eligible["impressions_90d"] >= 500) & (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20) & (eligible["ctr"] < 0.5)
low_engagement_visible = (eligible["sessions_90d"] >= 30) & (eligible["engagement_rate"] < 30)

rule_hits = pd.concat(
    [stale_visible, declining_with_demand, thin_visible, page_one_decay_risk, low_ctr_visible, low_engagement_visible],
    axis=1,
).sum(axis=1)

print("How many of the 6 reason-code rules fire per row (0 = matches none):")
print(rule_hits.value_counts().sort_index())
print()
pct_multi = (rule_hits >= 2).mean() * 100
print(f"{pct_multi:.1f}% of eligible rows trip 2+ rules simultaneously.")
print("A plain rule set can flag these rows, but has no principled way to rank them")
print("against each other -- which is precisely the gap a model fills.")


How many of the 6 reason-code rules fire per row (0 = matches none):
0    8976
1    9676
2    7191
3    3487
4     670
Name: count, dtype: int64

37.8% of eligible rows trip 2+ rules simultaneously.
A plain rule set can flag these rows, but has no principled way to rank them
against each other -- which is precisely the gap a model fills.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.